# MLflow Experiment Tracking

This notebook is used to add on MLflow experiment tracking to the Airbnb price predictor. Aims to log model runs, parameters, and evaluate metrics to create a single experiment dashboard to compare regression models. 

In [1]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
DATA_PATH = "../data/processed/clean_listings.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(37422, 22)


,id,price,log_price,host_is_superhost,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,...,beds,minimum_nights,maximum_nights,availability_365,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,amenities_count
0,2708,67.22,4.222738,1.0,Hollywood,34.09625,-118.34605,Private room in rental unit,Private room,1,...,1.0,30.0,1125.0,301,47,4.87,4.94,4.96,4.87,77
1,2732,213.56,5.368589,0.0,Santa Monica,34.00440,-118.48095,Private room in rental unit,Private room,1,...,1.0,1.0,27.0,350,24,4.41,4.58,4.91,4.22,19
2,6033,99.63,4.611450,0.0,Woodland Hills,34.16887,-118.64478,Entire bungalow,Entire home/apt,3,...,NaN,30.0,1125.0,270,19,4.38,4.00,4.65,4.29,32
3,6931,95.21,4.566533,1.0,Hollywood,34.09626,-118.34372,Private room in rental unit,Private room,1,...,1.0,30.0,1125.0,259,39,4.86,4.92,4.69,4.75,72
4,7874,113.00,4.736198,0.0,Bellflower,33.87687,-118.11444,Private room in home,Private room,2,...,1.0,1.0,730.0,103,26,4.77,4.88,4.77,4.77,22


In [3]:
target_column = "log_price"
excluded_from_features = ["id", "price", "log_price"]

X = df.drop(columns=excluded_from_features)
y = df[target_column]

print("X shape:", X.shape)
print("Y shape:", y.shape)

X shape: (37422, 19)
Y shape: (37422,)


In [4]:
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['host_is_superhost', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'availability_365', 'number_of_reviews', 'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location', 'review_scores_value', 'amenities_count']

Categorical features: ['neighbourhood_cleansed', 'property_type', 'room_type']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (29937, 19)
Test shape: (7485, 19)


In [6]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
dense_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

dense_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", dense_categorical_transformer, categorical_features),
    ]
)

In [8]:
def evaluate_model(name, model, X_test, y_test):
    log_preds = model.predict(X_test)

    # Convert log preds back to orginal scale
    y_test_dollars = np.expm1(y_test)
    preds_dollars = np.expm1(log_preds)
    preds_dollars = np.maximum(preds_dollars, 0)

    log_rmse = np.sqrt(mean_squared_error(y_test, log_preds))
    dollar_mae = mean_absolute_error(y_test_dollars, preds_dollars)
    dollar_rmse = np.sqrt(mean_squared_error(y_test_dollars, preds_dollars))
    r2 = r2_score(y_test, log_preds)

    return {
        "model": name, 
        "log_rmse": log_rmse,
        "dollar_mae": dollar_mae,
        "dollar_rmse": dollar_rmse,
        "r2_log_price": r2,
    }

## MLflow set up

In [9]:
MLFLOW_DB_PATH = Path("../mlflow.db").resolve()

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
mlflow.set_experiment("airbnb_price_prediction")

print("MLflow tracking database:", MLFLOW_DB_PATH)

2026/08/14 18:12:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/14 18:12:16 INFO mlflow.store.db.utils: Updating database tables
2026/08/14 18:12:16 INFO mlflow.tracking.fluent: Experiment with name 'airbnb_price_prediction' does not exist. Creating a new experiment.


MLflow tracking database: /Users/lucalivigni/Desktop/Summer2026Projects/ml-project/mlflow.db


## Models

In [10]:
models = {
    "Median Baseline": {
        "model": DummyRegressor(strategy="median"),
        "params": {"strategy": "median"},
    }, 
    "Linear Regression": {
        "model": LinearRegression(),
        "params": {},
    },
    "Ridge": {
        "model": Ridge(alpha=1.0),
        "params": {"alpha": 1.0},
    },
    "Random Forest": {
        "model": RandomForestRegressor(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": 100,
            "max_depth": 20,
            "min_samples_leaf": 5,
            "random_state": 42,
        },
    },
    "Hist Gradient Boosting": {
        "model": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=42,
        ),
        "params": {
            "max_iter": 300,
            "learning_rate": 0.05,
            "max_leaf_nodes": 31,
            "random_state": 42,
        },
    },
}

## Train and log models

In [11]:
mlflow.end_run()

mlflow_results = []

for model_name, model_info in models.items():
    with mlflow.start_run(run_name=model_name):

        if model_name == "Hist Gradient Boosting":
            model_preprocessor = dense_preprocessor
        else:
            model_preprocessor = preprocessor
        
        pipeline = Pipeline(
            steps = [ 
                ("preprocessor", model_preprocessor),
                ("model", model_info["model"]),
            ]
        )

        pipeline.fit(X_train, y_train)
        metrics = evaluate_model(model_name, pipeline, X_test, y_test)
        
        mlflow.log_param("model_name", model_name)
        mlflow.log_params(model_info["params"])
        mlflow.log_metrics({
            "log_rmse": metrics["log_rmse"],
            "dollar_mae": metrics["dollar_mae"],
            "dollar_rmse": metrics["dollar_rmse"],
            "r2_log_price": metrics["r2_log_price"],
        })

        mlflow_results.append(metrics)

mlflow_results_df = pd.DataFrame(mlflow_results).sort_values("log_rmse")
mlflow_results_df

,model,log_rmse,dollar_mae,dollar_rmse,r2_log_price
4,Hist Gradient Boosting,0.336277,91.367438,190.685782,0.851770
3,Random Forest,0.350174,94.089006,197.892375,0.839265
2,Ridge,0.402065,109.566215,230.130041,0.788098
1,Linear Regression,0.402806,109.668703,229.961961,0.787317
0,Median Baseline,0.873487,213.833522,385.681875,-0.000125


In [12]:
mlflow_results_df.to_csv("../reports/mlflow_model_results.csv", index=False)

mlflow_results_df.round({
    "log_rmse": 3,
    "dollar_mae": 2,
    "dollar_rmse": 2,
    "r2_log_price": 3,
})

,model,log_rmse,dollar_mae,dollar_rmse,r2_log_price
4,Hist Gradient Boosting,0.336,91.37,190.69,0.852
3,Random Forest,0.350,94.09,197.89,0.839
2,Ridge,0.402,109.57,230.13,0.788
1,Linear Regression,0.403,109.67,229.96,0.787
0,Median Baseline,0.873,213.83,385.68,-0.000
